# VoxCPM Text-to-Voice (Colab)

Ready-to-run notebook for Reelday.ph marketing voiceover using [VoxCPM](https://github.com/OpenBMB/VoxCPM) (Apache-2.0, free for commercial use).

**Before you run:** set the GPU runtime.
`Runtime` → `Change runtime type` → Hardware accelerator = **T4 GPU** → Save.

Then run the cells top to bottom. Cell 1 (install) takes ~3–5 min on a fresh session and must be re-run each new session.

**Voices:** clone your own voice from a short reference clip — section 6. Delivery (energy, pace, emotion) is controlled by *how you record that clip*, not by text tags — this build reads inline `(style)` / `[description]` tags aloud rather than interpreting them.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('CUDA available:', torch.cuda.is_available())

## 2. Install VoxCPM (~3–5 min, re-run each session)

In [ ]:
!pip install -q voxcpm soundfile
print('Done. If you see dependency warnings, they are usually safe to ignore.')

## 3. Load the model (downloads weights on first run)

In [ ]:
from voxcpm import VoxCPM
import soundfile as sf
from IPython.display import Audio, display

model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=False,
)
print('Model loaded. Sample rate:', model.tts_model.sample_rate)

## 4. Generate voiceover (default model voice)

Edit `TEXT` below, run the cell, then play / download the result.
- `cfg_value`: higher = sticks closer to a reference style (2.0 is a good default).
- `inference_timesteps`: higher = better quality but slower (10 is a good default).

In [ ]:
TEXT = "Capture every moment of your big day with Reelday. Real videographers, edited reels, delivered fast."
OUTPUT_FILE = "voiceover.wav"

wav = model.generate(
    text=TEXT,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUTPUT_FILE, wav, model.tts_model.sample_rate)
print('Saved', OUTPUT_FILE)
display(Audio(OUTPUT_FILE))

## 5. Download the audio file

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## 6. Use YOUR OWN voice (cloning)

Record or upload a short, clean reference clip (**~5–15 seconds**, no music/noise) of the voice you want to clone. Only clone a voice you own or have permission to use.

**Tip:** for best fidelity, type the exact words spoken in the clip into `PROMPT_TEXT`.

Run the upload cell first, then the cloning cell.

In [ ]:
# Upload your reference voice clip (.wav recommended)
from google.colab import files
uploaded = files.upload()
REF_WAV = list(uploaded.keys())[0]
print('Using reference:', REF_WAV)

In [ ]:
# Exact transcript of what's said in the reference clip (improves fidelity).
PROMPT_TEXT = "Type here exactly what is spoken in your reference clip."

TEXT = "Capture every moment of your big day with Reelday. Book free today."
OUT = "cloned_voiceover.wav"

wav = model.generate(
    text=TEXT,
    prompt_wav_path=REF_WAV,
    prompt_text=PROMPT_TEXT,
    reference_wav_path=REF_WAV,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUT, wav, model.tts_model.sample_rate)
print('Saved', OUT)
display(Audio(OUT))
files.download(OUT)

## 6b. Tagalog / Taglish test (in your cloned voice)

Run section 6 first (upload your clip + set `PROMPT_TEXT`). This reuses your `REF_WAV` and `PROMPT_TEXT` and renders several Filipino / Taglish marketing lines so you can judge pronunciation before committing to a full batch.

**Controlling delivery:** this VoxCPM build does **not** support inline `(style)` tags — it just reads them aloud. Instead, the output's energy and pace come from **how you record your reference clip in section 6**. Want upbeat ads? Record your reference clip upbeat. Want calm? Record it calm. The clone copies your delivery, not just your timbre.

VoxCPM is tuned mainly on English/Chinese, so listen for odd stress on the pure-Tagalog lines — Taglish usually fares better.

In [ ]:
TAGALOG_LINES = {
    "tl_1": "Sa Reelday, hindi lang litrato — buong kwento ng kasal mo, naka-video.",
    "tl_2": "Mula sa I do hanggang sa first dance, kami ang kukuha ng bawat sandali.",
    "taglish_1": "Libre mag-start sa Reelday! Real videographers, edited reels, delivered fast.",
    "taglish_2": "Book na ang Reelday mo today — para hindi ka mawalan ng kahit isang moment.",
}

for name, line in TAGALOG_LINES.items():
    wav = model.generate(
        text=line,
        prompt_wav_path=REF_WAV,
        prompt_text=PROMPT_TEXT,
        reference_wav_path=REF_WAV,
        cfg_value=2.0,
        inference_timesteps=10,
    )
    fname = f"{name}.wav"
    sf.write(fname, wav, model.tts_model.sample_rate)
    print(name, '-', line)
    display(Audio(fname))

## 6c. Voice library — multiple mood clips + cfg control

Build a small library of your own voice in different moods, then pick one per generation. This is the real "styling" workflow for this VoxCPM build.

**Step 1 — record a few reference clips** (~5–15s each, clean), one per mood, e.g.:
- `upbeat.wav` — energetic, smiling → hooks / CTAs
- `calm.wav` — soft, warm → emotional wedding spots
- `narration.wav` — neutral, steady → explainer voiceover

**Step 2 — run the upload cell**, then fill in the `VOICES` map with each file's exact transcript.

**Step 3 — pick a `VOICE` + `CFG` and generate.**
- `CFG` higher (2.5–3.0) = sticks tightly to that clip's tone; lower (1.3–1.8) = more expressive but can drift.
- Set `DENOISE = True` if a clip has background noise.

In [ ]:
# Step 2a: upload all your mood clips at once (select multiple in the dialog)
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Step 2b: map each mood to its file + the EXACT transcript of that clip.
VOICES = {
    "upbeat":    {"wav": "upbeat.wav",    "text": "Exact words spoken in your upbeat clip."},
    "calm":      {"wav": "calm.wav",      "text": "Exact words spoken in your calm clip."},
    "narration": {"wav": "narration.wav", "text": "Exact words spoken in your narration clip."},
}

# Step 3: pick a mood + tune, then generate.
VOICE   = "upbeat"        # which entry from VOICES above
CFG     = 2.0             # 2.5-3.0 = tighter to clip tone; 1.3-1.8 = more expressive
DENOISE = False          # True if the reference clip has background noise

TEXT = "Libre mag-start sa Reelday! Real videographers, edited reels, delivered fast."
OUT  = f"{VOICE}_output.wav"

v = VOICES[VOICE]
wav = model.generate(
    text=TEXT,
    prompt_wav_path=v["wav"],
    prompt_text=v["text"],
    reference_wav_path=v["wav"],
    cfg_value=CFG,
    inference_timesteps=10,
    denoise=DENOISE,
)
sf.write(OUT, wav, model.tts_model.sample_rate)
print(f"Saved {OUT}  (voice={VOICE}, cfg={CFG})")
display(Audio(OUT))
files.download(OUT)

## 7. Default model voice (no cloning)

Generates speech in VoxCPM's own built-in voice — no reference clip needed. Useful for a quick test or if you don't want to clone.

**Note:** inline voice descriptions (e.g. `[warm Filipina narrator]`) are **not** supported in this build — the model reads them aloud. To control the voice, use cloning (section 6) with a reference clip recorded in the style you want.

In [ ]:
TEXT = "Your wedding deserves more than photos. With Reelday, every moment becomes a reel."
OUT = "default_voice.wav"

wav = model.generate(
    text=TEXT,
    cfg_value=2.0,
    inference_timesteps=10,
)
sf.write(OUT, wav, model.tts_model.sample_rate)
print('Saved', OUT)
display(Audio(OUT))
files.download(OUT)

## 8. Batch generation (optional)

Generate several clips at once — useful for multiple ad variations. Add `prompt_wav_path` / `prompt_text` / `reference_wav_path` to each call if you want them in your cloned voice.

In [ ]:
LINES = {
    "hook_1": "Your wedding deserves more than photos.",
    "hook_2": "From I do to the dance floor — we film it all.",
    "cta":    "Book your Reelday today. It's free to start.",
}

for name, line in LINES.items():
    wav = model.generate(text=line, cfg_value=2.0, inference_timesteps=10)
    fname = f"{name}.wav"
    sf.write(fname, wav, model.tts_model.sample_rate)
    print(name)
    display(Audio(fname))